# Notebook 4: Quantum Interference

This is the centerpiece of our series. Everything so far has been building toward a
single idea:

> **Quantum amplitudes are signed (or complex) numbers that are summed before squaring.
> This means paths to the same outcome can reinforce (constructive interference) or
> cancel (destructive interference) -- something impossible with classical probabilities.**

In this notebook:

1. The Hadamard interference demo: step-by-step through $H$, $HH$, $HZH$.
2. Path amplitude sums: constructive and destructive interference.
3. Classical vs quantum combination rules.
4. **Demo B**: explicit path contributions showing why $HH|0\rangle = |0\rangle$ and $HZH|0\rangle = |1\rangle$.

In [ ]:
import sys
sys.path.insert(0, '../src')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from quantum_demo.interference import (
    hadamard_interference_demo,
    path_amplitude_sum,
    compare_probability_vs_amplitude_combination,
)
from quantum_demo.states import amplitudes_to_probabilities
from quantum_demo.gates import H, Z, apply_gate
from quantum_demo.linalg import ket
from quantum_demo.viz.static_plots import (
    plot_probabilities,
    plot_real_amplitudes,
)

---

## 1. The Hadamard interference demo

The Hadamard gate $H$ is the simplest setting for interference. Recall:

$$H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$

It maps $|0\rangle \to |+\rangle$ and $|1\rangle \to |-\rangle$. When applied twice,
$HH = I$, so $HH|0\rangle = |0\rangle$. But *why*? The answer is interference.

Let's walk through every intermediate state.

In [ ]:
demo = hadamard_interference_demo()

# Print all intermediate states and probabilities
print("Step 0: |0>")
print(f"  state = {demo['ket0']}")
print()
print("Step 1: H|0> = |+>  (equal superposition)")
print(f"  state = {demo['H_ket0']}")
print(f"  probs = {demo['H_ket0_probs']}")
print()
print("Step 2: HH|0>  (second Hadamard -> back to |0>)")
print(f"  state = {np.real(demo['HH_ket0']).round(10)}")
print(f"  probs = {demo['HH_ket0_probs']}")
print()
print("Step 3: ZH|0>  (phase flip on the superposition)")
print(f"  state = {demo['Z_H_ket0']}")
print(f"  probs = {demo['Z_H_ket0_probs']}")
print()
print("Step 4: HZH|0>  (Hadamard after phase flip -> |1>!)")
print(f"  state = {np.real(demo['HZH_ket0']).round(10)}")
print(f"  probs = {demo['HZH_ket0_probs']}")

### Visualizing each step

Let's see the amplitudes and probabilities at each stage.

In [ ]:
labels = ['|0>', '|1>']

fig, axes = plt.subplots(2, 4, figsize=(14, 6))

steps = [
    ('|0>', demo['ket0']),
    ('H|0>', demo['H_ket0']),
    ('HH|0>', demo['HH_ket0']),
    ('HZH|0>', demo['HZH_ket0']),
]

for col, (title, state) in enumerate(steps):
    # Top row: amplitudes
    plot_real_amplitudes(state, labels=labels, title=f'{title} amplitudes', ax=axes[0, col])
    # Bottom row: probabilities
    plot_probabilities(amplitudes_to_probabilities(state), labels=labels,
                       title=f'{title} probabilities', ax=axes[1, col])

fig.suptitle('Hadamard Interference: Amplitudes (top) vs Probabilities (bottom)',
             fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

**Key observations:**

- After the first $H$, both $|0\rangle$ and $|1\rangle$ have amplitude $1/\sqrt{2}$,
  giving 50/50 probabilities.
- After the second $H$ (i.e., $HH|0\rangle$), we are back to $|0\rangle$ with certainty.
  The two paths to $|1\rangle$ had amplitudes $+1/2$ and $-1/2$ that **cancelled** (destructive interference),
  while the two paths to $|0\rangle$ had amplitudes $+1/2$ and $+1/2$ that **reinforced** (constructive interference).
- Inserting a $Z$ gate between the two Hadamards flips the sign of $|1\rangle$'s amplitude.
  Now the paths to $|0\rangle$ cancel and the paths to $|1\rangle$ reinforce, giving $|1\rangle$ with certainty.

The $Z$ gate changed **no probabilities at all** (Steps 1 and 3 have identical probability bars),
but it completely redirected the interference pattern.

---

## 2. Path amplitude sums

The core quantum rule is:

$$\text{total amplitude} = \sum_{\text{paths}} a_k, \qquad
\text{probability} = \left|\sum_k a_k\right|^2$$

Compare this with the classical rule:

$$\text{probability} = \sum_{\text{paths}} p_k$$

The difference is profound: quantum amplitudes are summed *before* squaring, so they
can cancel (if they have opposite signs) or reinforce (if they have the same sign).
Classical probabilities are summed directly and can never cancel.

In [ ]:
# Constructive interference: same-sign amplitudes
amp, prob = path_amplitude_sum([0.5, 0.5])
print("=== Constructive Interference ===")
print(f"Path amplitudes:  [+0.5, +0.5]")
print(f"Total amplitude:  {amp}")
print(f"Probability:      |{amp}|^2 = {prob}")
print()

In [ ]:
# Destructive interference: opposite-sign amplitudes
amp, prob = path_amplitude_sum([0.5, -0.5])
print("=== Destructive Interference ===")
print(f"Path amplitudes:  [+0.5, -0.5]")
print(f"Total amplitude:  {amp}")
print(f"Probability:      |{amp}|^2 = {prob}")

In [ ]:
# Partial interference: amplitudes don't fully cancel
amp, prob = path_amplitude_sum([0.6, -0.3])
print("=== Partial Interference ===")
print(f"Path amplitudes:  [+0.6, -0.3]")
print(f"Total amplitude:  {amp}")
print(f"Probability:      |{amp}|^2 = {prob:.4f}")
print(f"\nClassical sum of |a|^2: {0.6**2 + 0.3**2:.4f}  (larger -- no cancellation)")

In [ ]:
# Complex amplitudes can interfere too
amp, prob = path_amplitude_sum([0.5, 0.5j])
print("=== Complex Interference ===")
print(f"Path amplitudes:  [0.5, 0.5j]")
print(f"Total amplitude:  {amp}")
print(f"Probability:      |{amp}|^2 = {prob:.4f}")
print(f"Sum of individual |a|^2:    {0.5**2 + 0.5**2:.4f}")
print(f"\nThe probability is the same because the amplitudes are orthogonal in the complex plane.")

---

## 3. Classical vs quantum combination rules

Let's use the built-in comparison function that crystallizes the difference.

In [ ]:
comparison = compare_probability_vs_amplitude_combination()

print("=" * 65)
print("CLASSICAL PROBABILITY")
print("=" * 65)
c = comparison['classical_probs']
print(f"Two paths, each with probability {c['paths'][0]}")
print(f"Total probability = {c['paths'][0]} + {c['paths'][1]} = {c['total']}")
print()

print("=" * 65)
print("QUANTUM -- CONSTRUCTIVE INTERFERENCE")
print("=" * 65)
qc = comparison['quantum_constructive']
print(f"Two paths, each with amplitude {qc['amplitudes'][0]}")
print(f"Total amplitude = {qc['amplitudes'][0]} + {qc['amplitudes'][1]} = {sum(qc['amplitudes'])}")
print(f"Probability = |{sum(qc['amplitudes'])}|^2 = {qc['probability']}")
print()

print("=" * 65)
print("QUANTUM -- DESTRUCTIVE INTERFERENCE")
print("=" * 65)
qd = comparison['quantum_destructive']
print(f"Two paths, amplitudes {qd['amplitudes'][0]} and {qd['amplitudes'][1]}")
print(f"Total amplitude = {qd['amplitudes'][0]} + ({qd['amplitudes'][1]}) = {sum(qd['amplitudes'])}")
print(f"Probability = |{sum(qd['amplitudes'])}|^2 = {qd['probability']}")

In [ ]:
print()
print(comparison['explanation'])

In [ ]:
# Visual comparison: classical addition vs quantum constructive vs quantum destructive
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Classical
axes[0].bar(['Path 1', 'Path 2', 'Total'], 
            [0.25, 0.25, 0.50],
            color=['#4C72B0', '#4C72B0', '#55A868'], edgecolor='black')
axes[0].set_ylim(0, 1.15)
axes[0].set_title('Classical\n(probs add)')
axes[0].set_ylabel('Probability')
for i, v in enumerate([0.25, 0.25, 0.50]):
    axes[0].text(i, v + 0.03, f'{v:.2f}', ha='center', fontsize=10)

# Quantum constructive
axes[1].bar(['Path 1', 'Path 2', '|Sum|^2'], 
            [0.25, 0.25, 1.00],
            color=['#4C72B0', '#4C72B0', '#55A868'], edgecolor='black')
axes[1].set_ylim(0, 1.15)
axes[1].set_title('Quantum Constructive\n(amps reinforce)')
for i, v in enumerate([0.25, 0.25, 1.00]):
    axes[1].text(i, v + 0.03, f'{v:.2f}', ha='center', fontsize=10)

# Quantum destructive
axes[2].bar(['Path 1', 'Path 2', '|Sum|^2'], 
            [0.25, 0.25, 0.00],
            color=['#4C72B0', '#C44E52', '#55A868'], edgecolor='black')
axes[2].set_ylim(0, 1.15)
axes[2].set_title('Quantum Destructive\n(amps cancel)')
for i, v in enumerate([0.25, 0.25, 0.00]):
    axes[2].text(i, v + 0.03, f'{v:.2f}', ha='center', fontsize=10)

fig.suptitle('The same path "weights" produce different total probabilities',
             fontsize=13, y=1.04)
fig.tight_layout()
plt.show()

The three panels make the point starkly:

- **Classical**: Two paths each contributing probability 0.25 always give total 0.50.
- **Quantum (constructive)**: Two paths each contributing amplitude 0.5 (both positive)
  give total amplitude 1.0, hence probability 1.00 -- *double* the classical result.
- **Quantum (destructive)**: Two paths contributing amplitudes +0.5 and -0.5
  give total amplitude 0.0, hence probability 0.00 -- *zero*, despite each path
  individually having probability 0.25.

This is impossible classically. It is the signature of quantum interference.

---

## 4. Demo B: Explicit path contributions in $HH|0\rangle$ and $HZH|0\rangle$

Let's trace the exact path amplitudes through the Hadamard gate to see interference
in action at the matrix level.

### Setup

Start with $|0\rangle$. After the first $H$, we have:

$$H|0\rangle = \frac{1}{\sqrt{2}}|0\rangle + \frac{1}{\sqrt{2}}|1\rangle$$

Now apply $H$ again. Each term in the superposition creates two sub-paths:

| Source | Via $H$ to $|0\rangle$ | Via $H$ to $|1\rangle$ |
|--------|------------------------|------------------------|
| $\frac{1}{\sqrt{2}}|0\rangle$ | $\frac{1}{\sqrt{2}} \cdot \frac{1}{\sqrt{2}} = +\frac{1}{2}$ | $\frac{1}{\sqrt{2}} \cdot \frac{1}{\sqrt{2}} = +\frac{1}{2}$ |
| $\frac{1}{\sqrt{2}}|1\rangle$ | $\frac{1}{\sqrt{2}} \cdot \frac{1}{\sqrt{2}} = +\frac{1}{2}$ | $\frac{1}{\sqrt{2}} \cdot (-\frac{1}{\sqrt{2}}) = -\frac{1}{2}$ |

The minus sign in the bottom-right entry comes from $H_{11} = -1/\sqrt{2}$.

In [ ]:
# Let's compute the path contributions explicitly
h = 1 / np.sqrt(2)  # shorthand

# After first H, amplitudes in the superposition
a0 = h   # amplitude of |0> component
a1 = h   # amplitude of |1> component

# Second H: matrix entries H[i,j]
print("Hadamard matrix:")
print(np.real(H).round(6))
print()

# Paths to |0> in the final state:
path_0_from_0 = a0 * H[0, 0]  # |0> -> H -> |0>
path_0_from_1 = a1 * H[0, 1]  # |1> -> H -> |0>

# Paths to |1> in the final state:
path_1_from_0 = a0 * H[1, 0]  # |0> -> H -> |1>
path_1_from_1 = a1 * H[1, 1]  # |1> -> H -> |1>

print("=== Paths to |0> ===")
print(f"  From |0> component: {a0:.4f} * H[0,0] = {a0:.4f} * {np.real(H[0,0]):.4f} = {np.real(path_0_from_0):.4f}")
print(f"  From |1> component: {a1:.4f} * H[0,1] = {a1:.4f} * {np.real(H[0,1]):.4f} = {np.real(path_0_from_1):.4f}")
total_0 = path_0_from_0 + path_0_from_1
print(f"  SUM = {np.real(total_0):.4f}  => P(0) = {abs(total_0)**2:.4f}")

print()
print("=== Paths to |1> ===")
print(f"  From |0> component: {a0:.4f} * H[1,0] = {a0:.4f} * {np.real(H[1,0]):.4f} = {np.real(path_1_from_0):.4f}")
print(f"  From |1> component: {a1:.4f} * H[1,1] = {a1:.4f} * {np.real(H[1,1]):.4f} = {np.real(path_1_from_1):.4f}")
total_1 = path_1_from_0 + path_1_from_1
print(f"  SUM = {np.real(total_1):.4f}  => P(1) = {abs(total_1)**2:.4f}")

For the path to $|0\rangle$: both contributions are $+1/2$, so they **add** to give amplitude $1.0$.
This is **constructive interference**.

For the path to $|1\rangle$: the contributions are $+1/2$ and $-1/2$, so they **cancel** to give amplitude $0.0$.
This is **destructive interference**.

Result: $HH|0\rangle = |0\rangle$ with certainty.

In [ ]:
# Verify with the direct computation
ket0 = ket(0, 2)
hh_ket0 = apply_gate(apply_gate(ket0, H), H)
print("HH|0> =", np.real(hh_ket0).round(10))
print("Confirmed:", np.allclose(hh_ket0, ket0))

### Now with a Z gate in the middle: $HZH|0\rangle$

The $Z$ gate flips the sign of the $|1\rangle$ amplitude:
$Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$.

After $H|0\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$, the $Z$ gate gives
$ZH|0\rangle = \frac{1}{\sqrt{2}}(|0\rangle - |1\rangle)$.

The probabilities are unchanged (still 50/50), but the sign of the $|1\rangle$ amplitude
has flipped. This changes the interference pattern in the final $H$.

In [ ]:
# After ZH|0>: amplitudes are +1/sqrt(2) and -1/sqrt(2)
a0_z = h     # amplitude of |0> component (unchanged by Z)
a1_z = -h    # amplitude of |1> component (sign flipped by Z)

# Paths to |0> in the final state after the second H:
path_0_from_0_z = a0_z * H[0, 0]
path_0_from_1_z = a1_z * H[0, 1]

# Paths to |1> in the final state:
path_1_from_0_z = a0_z * H[1, 0]
path_1_from_1_z = a1_z * H[1, 1]

print("After ZH|0>: amplitudes are", [a0_z, a1_z])
print()
print("=== Paths to |0> ===")
print(f"  From |0> component: {a0_z:.4f} * {np.real(H[0,0]):.4f} = {np.real(path_0_from_0_z):.4f}")
print(f"  From |1> component: {a1_z:.4f} * {np.real(H[0,1]):.4f} = {np.real(path_0_from_1_z):.4f}")
total_0_z = path_0_from_0_z + path_0_from_1_z
print(f"  SUM = {np.real(total_0_z):.4f}  => P(0) = {abs(total_0_z)**2:.4f}")

print()
print("=== Paths to |1> ===")
print(f"  From |0> component: {a0_z:.4f} * {np.real(H[1,0]):.4f} = {np.real(path_1_from_0_z):.4f}")
print(f"  From |1> component: {a1_z:.4f} * {np.real(H[1,1]):.4f} = {np.real(path_1_from_1_z):.4f}")
total_1_z = path_1_from_0_z + path_1_from_1_z
print(f"  SUM = {np.real(total_1_z):.4f}  => P(1) = {abs(total_1_z)**2:.4f}")

The $Z$ gate reversed which outcome gets constructive vs destructive interference:

- Now the paths to $|0\rangle$ are $+1/2$ and $-1/2$ -- they **cancel** (destructive).
- The paths to $|1\rangle$ are $+1/2$ and $+1/2$ -- they **reinforce** (constructive).

Result: $HZH|0\rangle = |1\rangle$ with certainty.

In [ ]:
# Verify
ket1 = ket(1, 2)
hzh_ket0 = apply_gate(apply_gate(apply_gate(ket0, H), Z), H)
print("HZH|0> =", np.real(hzh_ket0).round(10))
print("Confirmed:", np.allclose(hzh_ket0, ket1))

### Visualizing the path contributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# --- Left panel: HH|0> paths ---
bar_labels = ['via |0>\n(+1/2)', 'via |1>\n(+1/2)', 'Sum\n(+1.0)', 'via |0>\n(+1/2)', 'via |1>\n(-1/2)', 'Sum\n(0.0)']
values_hh = [0.5, 0.5, 1.0, 0.5, -0.5, 0.0]
colors_hh = ['#55A868', '#55A868', '#4C72B0', '#55A868', '#C44E52', '#4C72B0']

x_pos = [0, 1, 2, 4, 5, 6]
axes[0].bar(x_pos, values_hh, color=colors_hh, edgecolor='black', linewidth=0.5)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(bar_labels, fontsize=8)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_ylabel('Amplitude')
axes[0].set_title('HH|0>: paths to |0> (left) and |1> (right)')
axes[0].set_ylim(-0.8, 1.2)

# Add bracket labels
axes[0].annotate('Constructive', xy=(1, 1.05), fontsize=9, ha='center', color='#4C72B0', fontweight='bold')
axes[0].annotate('Destructive', xy=(5, 0.15), fontsize=9, ha='center', color='#C44E52', fontweight='bold')

# --- Right panel: HZH|0> paths ---
values_hzh = [0.5, -0.5, 0.0, 0.5, 0.5, 1.0]
colors_hzh = ['#55A868', '#C44E52', '#4C72B0', '#55A868', '#55A868', '#4C72B0']

axes[1].bar(x_pos, values_hzh, color=colors_hzh, edgecolor='black', linewidth=0.5)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(bar_labels, fontsize=8)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_ylabel('Amplitude')
axes[1].set_title('HZH|0>: paths to |0> (left) and |1> (right)')
axes[1].set_ylim(-0.8, 1.2)

axes[1].annotate('Destructive', xy=(1, 0.15), fontsize=9, ha='center', color='#C44E52', fontweight='bold')
axes[1].annotate('Constructive', xy=(5, 1.05), fontsize=9, ha='center', color='#4C72B0', fontweight='bold')

fig.suptitle('Path Amplitude Contributions: Z gate swaps the interference pattern',
             fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

---

## 5. Using `path_amplitude_sum` for the HH and HZH cases

In [ ]:
# HH|0>: paths to |0>
amp_0_hh, prob_0_hh = path_amplitude_sum([0.5, 0.5])
# HH|0>: paths to |1>
amp_1_hh, prob_1_hh = path_amplitude_sum([0.5, -0.5])

print("HH|0>:")
print(f"  Paths to |0>: [+1/2, +1/2] -> amplitude {amp_0_hh}, P = {prob_0_hh}")
print(f"  Paths to |1>: [+1/2, -1/2] -> amplitude {amp_1_hh}, P = {prob_1_hh}")
print()

# HZH|0>: paths to |0>
amp_0_hzh, prob_0_hzh = path_amplitude_sum([0.5, -0.5])
# HZH|0>: paths to |1>
amp_1_hzh, prob_1_hzh = path_amplitude_sum([0.5, 0.5])

print("HZH|0>:")
print(f"  Paths to |0>: [+1/2, -1/2] -> amplitude {amp_0_hzh}, P = {prob_0_hzh}")
print(f"  Paths to |1>: [+1/2, +1/2] -> amplitude {amp_1_hzh}, P = {prob_1_hzh}")

---

## 6. Summary: why interference matters

| Concept | Classical | Quantum |
|---------|-----------|----------|
| **What flows along paths** | Probability (non-negative) | Amplitude (signed/complex) |
| **Combination rule** | Add probabilities | Add amplitudes, then square |
| **Can paths cancel?** | No -- probabilities only grow | Yes -- destructive interference |
| **Can paths reinforce beyond individual contributions?** | No -- maximum is the sum | Yes -- constructive interference |

### The big picture

Quantum computing works by arranging amplitudes so that:

- Paths leading to **wrong answers** interfere destructively (cancel out).
- Paths leading to **correct answers** interfere constructively (reinforce).

After this amplitude engineering, a measurement is overwhelmingly likely to
return the correct answer. This is the fundamental mechanism behind algorithms
like Grover's search and the quantum Fourier transform.

The $HZH$ example is the simplest possible instance of this idea: a single
sign flip ($Z$) completely redirects the interference pattern, steering the
system from $|0\rangle$ to $|1\rangle$ with certainty.